# 01 · Problema y datos
**Proyecto Integrador – Data Mining Tools (CC209)** · TP1

Este notebook cubre el **Paso 1** (descarga y registro de datos) y el **Paso 2** (definición del problema).

## 0. Configuración
Semilla global y rutas del proyecto, usadas por todos los notebooks.

In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_INTERIM = ROOT / "data" / "interim"
DATA_PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"

for d in (DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

ROOT: D:\TP-TF-Data_Mining
pandas 3.0.3 | numpy 2.4.4


## 1. Descarga y registro de datos

**Fuente:** King County House Sales, Kaggle (`harlfoxem/housesalesprediction`).
Ventas de viviendas en el condado de King (Seattle, WA, EE. UU.).

El archivo original se copia a `data/raw/` y **no se modifica**: toda limpieza se hará sobre copias.

In [2]:
RAW_FILE = DATA_RAW / "kc_house_data.csv"

if not RAW_FILE.exists():
    import kagglehub
    src_dir = Path(kagglehub.dataset_download("harlfoxem/housesalesprediction"))
    shutil.copy(src_dir / "kc_house_data.csv", RAW_FILE)
    print("Descargado y copiado a", RAW_FILE)
else:
    print("Ya existe:", RAW_FILE)

Ya existe: D:\TP-TF-Data_Mining\data\raw\kc_house_data.csv


In [3]:
df = pd.read_csv(RAW_FILE)
print(f"Observaciones: {df.shape[0]:,} | Variables: {df.shape[1]}")
df.head()

Observaciones: 21,613 | Variables: 21


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  str    
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long           21

### 1.1 Datos generales

In [5]:
fechas = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S")

resumen = pd.Series({
    "Observaciones": f"{len(df):,}",
    "Variables": df.shape[1],
    "Periodo de ventas": f"{fechas.min():%Y-%m-%d} a {fechas.max():%Y-%m-%d}",
    "Valores nulos (total)": int(df.isna().sum().sum()),
    "id únicos": f"{df['id'].nunique():,}",
    "id repetidos (ventas de una misma casa)": int(df["id"].duplicated().sum()),
    "Variable objetivo": "price (USD)",
})
resumen.to_frame("valor")

,valor
Observaciones,"21,613"
Variables,21
Periodo de ventas,2014-05-02 a 2015-05-27
Valores nulos (total),0
id únicos,"21,436"
id repetidos (ventas de una misma casa),177
Variable objetivo,price (USD)


**Qué muestran los datos**
- Cubre aproximadamente un año de ventas (mayo 2014 – mayo 2015), de un solo condado.
- No hay valores nulos. Eso no garantiza calidad: hay que revisar ceros que codifican "no aplica" (por ejemplo `yr_renovated`) y valores imposibles (Paso 5).
- Hay `id` repetidos: una misma vivienda aparece vendida más de una vez. Esto condiciona el split (Paso 6).

### 1.2 Diccionario de variables

In [6]:
diccionario = pd.DataFrame([
    ("id", "identificador", "Identificador de la vivienda (se repite si se vendió más de una vez)"),
    ("date", "fecha (texto)", "Fecha de la venta, formato YYYYMMDDT000000"),
    ("price", "numérica (USD)", "Precio de venta. Variable objetivo"),
    ("bedrooms", "numérica discreta", "Número de dormitorios"),
    ("bathrooms", "numérica", "Número de baños (fraccionario: 0.5 = medio baño)"),
    ("sqft_living", "numérica (pies²)", "Superficie habitable interior"),
    ("sqft_lot", "numérica (pies²)", "Superficie del terreno"),
    ("floors", "numérica", "Número de pisos"),
    ("waterfront", "binaria", "1 si da al agua, 0 si no"),
    ("view", "ordinal 0-4", "Calidad de la vista"),
    ("condition", "ordinal 1-5", "Estado de conservación"),
    ("grade", "ordinal 1-13", "Calidad de construcción y diseño según el sistema del condado"),
    ("sqft_above", "numérica (pies²)", "Superficie sobre el nivel del suelo"),
    ("sqft_basement", "numérica (pies²)", "Superficie del sótano (0 = sin sótano)"),
    ("yr_built", "año", "Año de construcción"),
    ("yr_renovated", "año", "Año de renovación (0 = nunca renovada)"),
    ("zipcode", "categórica", "Código postal. Es una categoría, no una cantidad"),
    ("lat", "numérica", "Latitud"),
    ("long", "numérica", "Longitud"),
    ("sqft_living15", "numérica (pies²)", "Superficie habitable promedio de las 15 viviendas vecinas más cercanas"),
    ("sqft_lot15", "numérica (pies²)", "Superficie de terreno promedio de las 15 viviendas vecinas más cercanas"),
], columns=["variable", "tipo", "descripción"])

# Verifica que el diccionario cubra exactamente las columnas del archivo
assert set(diccionario["variable"]) == set(df.columns), set(df.columns) ^ set(diccionario["variable"])
diccionario

,variable,tipo,descripción
0,id,identificador,Identificador de la vivienda (se repite si se ...
1,date,fecha (texto),"Fecha de la venta, formato YYYYMMDDT000000"
2,price,numérica (USD),Precio de venta. Variable objetivo
3,bedrooms,numérica discreta,Número de dormitorios
4,bathrooms,numérica,Número de baños (fraccionario: 0.5 = medio baño)
5,sqft_living,numérica (pies²),Superficie habitable interior
6,sqft_lot,numérica (pies²),Superficie del terreno
7,floors,numérica,Número de pisos
8,waterfront,binaria,"1 si da al agua, 0 si no"
9,view,ordinal 0-4,Calidad de la vista


### 1.3 Limitaciones conocidas del dataset
- **Alcance geográfico y temporal:** un solo condado y un solo año. Un modelo entrenado aquí no se puede extrapolar a otras ciudades ni a otros momentos del mercado.
- **Precio de cierre, no de oferta:** `price` es el precio al que se concretó la venta. No es un "valor justo", y las ventas atípicas (entre familiares, remates) no están marcadas.
- **Sin variables de entorno:** no hay colegios, criminalidad, transporte ni estado interior de la vivienda. Parte de la variación del precio queda sin explicar.
- **Ventas repetidas:** el mismo `id` puede aparecer más de una vez, con precios distintos.
- **Ceros ambiguos:** `yr_renovated = 0` y `sqft_basement = 0` significan "no aplica", no "año cero" ni "superficie nula".

### 1.4 Licencia y condiciones de uso
Publicado en Kaggle. **Verificar la licencia en la página del dataset** antes de la entrega y registrarla en `data/README.md`.
Los datos originales no se versionan en el repositorio (`data/raw/` está en `.gitignore`); se obtienen con el código de arriba.

---
## 2. Definición del problema

### 2.1 Contexto y necesidad
El precio de una vivienda depende de su tamaño, su calidad y, sobre todo, de su ubicación. Quien tasa, compra o vende necesita una referencia objetiva de cuánto *debería* costar una propiedad con determinadas características, y saber cuándo un precio se aleja de esa referencia.

Dos necesidades concretas en el mercado de King County:
1. **Estimar el precio de venta** de una vivienda a partir de sus características físicas y su ubicación.
2. **Identificar ventas atípicas**: operaciones cuyo precio se desvía mucho de lo esperado para propiedades similares. Pueden ser oportunidades, errores de registro o ventas con condiciones especiales.

### 2.2 Unidad de análisis
Una **venta de vivienda** (una fila del dataset). Una misma vivienda (`id`) puede aparecer más de una vez, con fechas y precios distintos.

### 2.3 Pregunta principal
> ¿Con qué error puede estimarse el precio de venta de una vivienda en King County a partir de sus características físicas y su ubicación, y qué ventas se desvían de forma anómala de lo que el modelo espera?

### 2.4 Tipo de problema
| Componente | Tipo | Salida |
|---|---|---|
| Principal | **Regresión supervisada** | Precio estimado (USD) |
| Secundario | **Detección de anomalías basada en residuos** | Ventas con precio real muy distinto del estimado |

La detección de anomalías depende de la calidad del modelo de regresión: un residuo grande puede indicar una venta atípica **o** una limitación del modelo. Esa distinción se discute al analizar los errores.

### 2.5 Utilidad esperada
- **Tasación de referencia:** una estimación del precio con un error cuantificado, para contrastar con el precio pedido.
- **Revisión de operaciones:** una lista priorizada de ventas que merecen una verificación.

### 2.6 Criterios de utilidad (definidos antes de modelar)
Para no ajustar el criterio a los resultados, se fija ahora. Son **propuestas iniciales, a validar con el grupo**:

| Criterio | Umbral propuesto | Por qué |
|---|---|---|
| Mejora sobre el baseline | El MAE del modelo debe ser claramente menor que el de `DummyRegressor` y que el de un baseline simple de dominio | Un modelo que no mejora a una regla trivial no aporta |
| Error relativo | Error porcentual mediano (MdAPE) en test **≤ 15 %** | Un error en dólares engaña entre casas de 150 mil y de 3 millones |
| Estabilidad | El desempeño en validación y en test debe ser similar | Descarta sobreajuste y fuga de información |
| Explicabilidad básica | Las variables que más pesan deben tener sentido en el dominio (tamaño, calidad, ubicación) | Un modelo que acierta por razones absurdas no es confiable |

El umbral de 15 % es una hipótesis de trabajo: se revisará con la evidencia del baseline y se documentará cualquier cambio con su motivo.

### 2.7 Qué NO se pretende afirmar
- El modelo estima el **precio de cierre observado**, no el valor "justo" ni el valor de mercado real.
- Una venta marcada como anómala **no** es un error ni un fraude: es una operación que el modelo no logra explicar con las variables disponibles.
- Los resultados valen para **King County en 2014-2015**. No se extrapolan a otras ciudades ni a otros momentos del mercado.
- No se establecen relaciones causales: que una variable pese en el modelo no significa que cambiarla modifique el precio.